In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from collections import Counter
import random
from math import floor

# ---------- Input / Output ----------
PROFILES_FILE = "survey_profiles.json"
STUDENT_STYLE_FILE = "survey_student_style.txt"
OUTPUT_BALANCED = "survey_profiles_clusterclip_balanced.json"

# ---------- Parameters ----------
N_CLUSTERS = 8                 # 6–10 is reasonable
SAMPLES_PER_GROUP = 40         # target per profile_strength ("weak","average","strong")
RANDOM_SEED = 42
MODEL_NAME = "all-MiniLM-L6-v2"  # Sentence-BERT variant

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---------- Load Data ----------
profiles = json.load(open(PROFILES_FILE, "r", encoding="utf-8"))

# sanity check
if not profiles or "profile_strength" not in profiles[0]:
    raise ValueError("survey_profiles.json missing 'profile_strength' in records.")

try:
    with open(STUDENT_STYLE_FILE, "r", encoding="utf-8") as f:
        style_lines = [l.strip() for l in f.readlines() if len(l.strip()) > 5]
except FileNotFoundError:
    style_lines = []

print(f"Loaded {len(profiles)} profiles and {len(style_lines)} style lines.")

df = pd.DataFrame(profiles).fillna("")

# ---------- Prepare combined textual + numeric representation ----------
def profile_to_text(p):
    # 'belonging' may not exist in your JSON -> use .get(..., "")
    fields = [
        f"GPA {p.get('cgpa', '')}",
        f"Confidence {p.get('confidence', '')}",
        f"Intent {p.get('continuation_intent', '')}",
        f"Work {p.get('work_hours', '')}",
        f"Sleep {p.get('sleep_cutback', '')}"
    ]
    return " | ".join(str(x) for x in fields)

df["combined_text"] = df.apply(profile_to_text, axis=1)

# add optional style context (1 line per row if available)
if style_lines:
    df["combined_text"] = (
        df["combined_text"] + " | " + np.random.choice(style_lines, size=len(df))
    )

# ---------- Generate Embeddings ----------
print("Encoding profiles with Sentence-BERT embeddings...")
encoder = SentenceTransformer(MODEL_NAME)
embeddings = encoder.encode(df["combined_text"].tolist(), show_progress_bar=True)

# ---------- Add normalized numeric features ----------
num_cols = ["cgpa", "confidence", "continuation_intent", "work_hours", "sleep_cutback"]
num_block = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
num_block = StandardScaler().fit_transform(num_block)

features = np.hstack([embeddings, num_block])

# ---------- Cluster Profiles ----------
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
clusters = kmeans.fit_predict(features)
df["cluster"] = clusters

print(f"Cluster sizes (global): {dict(Counter(clusters))}")

# ---------- Helper: stratified (by cluster) undersample/oversample to exact N ----------
def balance_group_by_cluster(group_df: pd.DataFrame, target_n: int, seed: int):
    """Return exactly target_n rows from group_df, preserving cluster proportions."""
    rng = np.random.default_rng(seed)
    # cluster distribution inside this group
    counts = group_df["cluster"].value_counts().to_dict()
    clusters_sorted = sorted(counts.keys())
    total = sum(counts.values())
    if total == 0:
        return group_df.sample(n=min(target_n, len(group_df)), replace=len(group_df) < target_n, random_state=seed)

    # proportional allocation (floor), then distribute remainder
    alloc = {c: floor(counts[c] / total * target_n) for c in clusters_sorted}
    allocated = sum(alloc.values())
    remainder = target_n - allocated
    # assign remainder to clusters with largest fractional parts
    fracs = {c: (counts[c] / total * target_n) - alloc[c] for c in clusters_sorted}
    for c in sorted(clusters_sorted, key=lambda x: fracs[x], reverse=True)[:remainder]:
        alloc[c] += 1

    # sample per cluster
    pieces = []
    for c in clusters_sorted:
        sub = group_df[group_df["cluster"] == c]
        need = alloc[c]
        if need <= 0:
            continue
        if len(sub) >= need:
            pieces.append(sub.sample(n=need, replace=False, random_state=seed))
        else:
            # oversample WITHIN the cluster to keep diversity across clusters
            take = sub.copy()
            if len(sub) > 0:
                extra = sub.sample(n=need - len(sub), replace=True, random_state=seed)
                pieces.append(pd.concat([take, extra], ignore_index=False))
            else:
                # if a cluster has zero rows (can happen rarely), skip; remainder will be short
                pass

    out = pd.concat(pieces) if pieces else group_df.head(0)
    # Top-off if some cluster had zero and we underfilled: fallback sample across group
    if len(out) < target_n and len(group_df) > 0:
        more = group_df.sample(
            n=target_n - len(out),
            replace=len(group_df) < (target_n - len(out)),
            random_state=seed,
        )
        out = pd.concat([out, more], ignore_index=False)
    # If we somehow overfilled due to rounding, trim
    if len(out) > target_n:
        out = out.sample(n=target_n, random_state=seed)
    return out

# ---------- Balance per profile_strength ----------
balanced_profiles = []
for label, group in df.groupby("profile_strength", dropna=False):
    if label not in {"weak", "average", "strong"}:
        # skip any unexpected labels
        continue
    target = SAMPLES_PER_GROUP
    balanced = balance_group_by_cluster(group, target, RANDOM_SEED)
    print(f"{label}: {len(group)} → {len(balanced)}")
    balanced_profiles.extend(balanced.to_dict(orient="records"))

# ---------- Save Balanced Dataset ----------
with open(OUTPUT_BALANCED, "w", encoding="utf-8") as f:
    json.dump(balanced_profiles, f, indent=2, ensure_ascii=False)

final_counts = Counter([p["profile_strength"] for p in balanced_profiles])
print(f"\nSaved semantically balanced dataset → {OUTPUT_BALANCED}")
print(f"Final per-group counts: {dict(final_counts)}")


/opt/anaconda3/envs/my_v_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 156 profiles and 783 style lines.
Encoding profiles with Sentence-BERT embeddings...


Batches: 100%|████████████████████████████████████| 5/5 [00:04<00:00,  1.01it/s]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Cluster sizes (global): {1: 33, 5: 29, 6: 8, 4: 1, 3: 25, 7: 16, 0: 33, 2: 11}
average: 16 → 40
strong: 59 → 40
weak: 81 → 40

Saved semantically balanced dataset → survey_profiles_clusterclip_balanced.json
Final per-group counts: {'average': 40, 'strong': 40, 'weak': 40}


/opt/anaconda3/envs/my_v_env/lib/python3.9/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
